<a href="https://colab.research.google.com/github/Md-Golam-Raiyhan/INSE-6320-Project/blob/main/risk_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# 1. Load dataset
# ==========================================
file_path = "metro_rail_project_dataset.csv"
df = pd.read_csv(file_path)

# output folder
output_dir = Path("monte_carlo_simulation_outputs")
output_dir.mkdir(exist_ok=True)

# ==========================================
# 2. Select key risk variables
# ==========================================
risk_vars = [
    "material_price_index",
    "labor_availability_index",
    "weather_delay_days",
    "permit_delay_days",
    "equipment_downtime_hours",
    "inflation_rate_annual",
    "change_orders"
]

# ==========================================
# 3. Estimate distribution parameters
# ==========================================
distribution_params = {}

for col in risk_vars:
    distribution_params[col] = {
        "Mean": df[col].mean(),
        "Std_Dev": df[col].std()
    }

params_df = pd.DataFrame(distribution_params).T.reset_index()
params_df.columns = ["Variable", "Mean", "Std_Dev"]
params_df[["Mean", "Std_Dev"]] = params_df[["Mean", "Std_Dev"]].round(4)

params_df.to_csv(output_dir / "table_distribution_parameters.csv", index=False)

print("\nDistribution Parameters")
print(params_df)

# ==========================================
# 4. Define baseline project cost
# ==========================================
baseline_cost = df["baseline_cost_musd"].mean()

# ==========================================
# 5. Monte Carlo Simulation
# ==========================================
num_simulations = 10000
simulated_costs = []

for _ in range(num_simulations):

    material = np.random.normal(
        distribution_params["material_price_index"]["Mean"],
        distribution_params["material_price_index"]["Std_Dev"]
    )

    labor = np.random.normal(
        distribution_params["labor_availability_index"]["Mean"],
        distribution_params["labor_availability_index"]["Std_Dev"]
    )

    weather = np.random.normal(
        distribution_params["weather_delay_days"]["Mean"],
        distribution_params["weather_delay_days"]["Std_Dev"]
    )

    permit = np.random.normal(
        distribution_params["permit_delay_days"]["Mean"],
        distribution_params["permit_delay_days"]["Std_Dev"]
    )

    equipment = np.random.normal(
        distribution_params["equipment_downtime_hours"]["Mean"],
        distribution_params["equipment_downtime_hours"]["Std_Dev"]
    )

    inflation = np.random.normal(
        distribution_params["inflation_rate_annual"]["Mean"],
        distribution_params["inflation_rate_annual"]["Std_Dev"]
    )

    change = np.random.normal(
        distribution_params["change_orders"]["Mean"],
        distribution_params["change_orders"]["Std_Dev"]
    )

    # simple analytical cost model
    total_cost = (
        baseline_cost
        + material * 5
        - labor * 2
        + weather * 0.3
        + permit * 0.4
        + equipment * 0.05
        + inflation * 4
        + change * 1.5
    )

    simulated_costs.append(total_cost)

simulated_costs = np.array(simulated_costs)

# save all runs if needed
pd.DataFrame({"Simulated_Total_Cost": simulated_costs}).to_csv(
    output_dir / "all_simulated_costs.csv",
    index=False
)

# ==========================================
# 6. Summary statistics
# ==========================================
results = {
    "Metric": [
        "Baseline Cost",
        "Mean Total Cost",
        "Standard Deviation",
        "Minimum Cost",
        "Maximum Cost",
        "P50",
        "P80",
        "P90",
        "P95",
        "Probability of Cost Overrun"
    ],
    "Value": [
        baseline_cost,
        np.mean(simulated_costs),
        np.std(simulated_costs),
        np.min(simulated_costs),
        np.max(simulated_costs),
        np.percentile(simulated_costs, 50),
        np.percentile(simulated_costs, 80),
        np.percentile(simulated_costs, 90),
        np.percentile(simulated_costs, 95),
        np.mean(simulated_costs > baseline_cost)
    ]
}

results_df = pd.DataFrame(results)
results_df["Value"] = results_df["Value"].round(4)
results_df.to_csv(output_dir / "table_monte_carlo_results.csv", index=False)

print("\nMonte Carlo Results")
print(results_df)

# ==========================================
# 7. Histogram of simulated cost
# ==========================================
plt.figure(figsize=(8, 5))
plt.hist(simulated_costs, bins=50, edgecolor="black")
plt.axvline(baseline_cost, linestyle="--", linewidth=2, label="Baseline Cost")
plt.title("Histogram of Simulated Total Project Cost")
plt.xlabel("Total Project Cost (Million USD)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "figure_histogram_simulated_cost.png", dpi=300)
plt.close()

# ==========================================
# 8. Cumulative distribution plot
# ==========================================
sorted_costs = np.sort(simulated_costs)
cdf = np.arange(1, len(sorted_costs) + 1) / len(sorted_costs)

plt.figure(figsize=(8, 5))
plt.plot(sorted_costs, cdf)
plt.axvline(np.percentile(simulated_costs, 80), linestyle="--", label="P80")
plt.axvline(np.percentile(simulated_costs, 90), linestyle="--", label="P90")
plt.title("Cumulative Distribution of Simulated Project Cost")
plt.xlabel("Total Project Cost (Million USD)")
plt.ylabel("Cumulative Probability")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "figure_cdf_simulated_cost.png", dpi=300)
plt.close()

print("\nAll Monte Carlo outputs saved in:", output_dir)


Distribution Parameters
                   Variable     Mean  Std_Dev
0      material_price_index   0.9948   0.0795
1  labor_availability_index   0.9332   0.0878
2        weather_delay_days   8.9682   3.0120
3         permit_delay_days   5.9091   2.2937
4  equipment_downtime_hours  27.7886  17.7452
5     inflation_rate_annual   0.0448   0.0114
6             change_orders   2.1591   1.4949

Monte Carlo Results
                        Metric     Value
0                Baseline Cost  113.3820
1              Mean Total Cost  126.3649
2           Standard Deviation    2.7573
3                 Minimum Cost  116.3356
4                 Maximum Cost  136.5588
5                          P50  126.3306
6                          P80  128.6722
7                          P90  129.8696
8                          P95  130.9270
9  Probability of Cost Overrun    1.0000

All Monte Carlo outputs saved in: monte_carlo_simulation_outputs


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# 1. Load dataset
# ==========================================
file_path = "metro_rail_project_dataset.csv"
df = pd.read_csv(file_path)

# output folder
output_dir = Path("sensitivity_analysis_outputs")
output_dir.mkdir(exist_ok=True)

# ==========================================
# 2. Select key risk variables
# ==========================================
risk_vars = [
    "material_price_index",
    "labor_availability_index",
    "weather_delay_days",
    "permit_delay_days",
    "equipment_downtime_hours",
    "inflation_rate_annual",
    "change_orders"
]

# ==========================================
# 3. Estimate distribution parameters
# ==========================================
distribution_params = {}

for col in risk_vars:
    distribution_params[col] = {
        "Mean": df[col].mean(),
        "Std_Dev": df[col].std()
    }

# baseline cost
baseline_cost = df["baseline_cost_musd"].mean()

# ==========================================
# 4. Monte Carlo simulation with input storage
# ==========================================
num_simulations = 10000
simulation_rows = []

for _ in range(num_simulations):

    material = np.random.normal(
        distribution_params["material_price_index"]["Mean"],
        distribution_params["material_price_index"]["Std_Dev"]
    )

    labor = np.random.normal(
        distribution_params["labor_availability_index"]["Mean"],
        distribution_params["labor_availability_index"]["Std_Dev"]
    )

    weather = np.random.normal(
        distribution_params["weather_delay_days"]["Mean"],
        distribution_params["weather_delay_days"]["Std_Dev"]
    )

    permit = np.random.normal(
        distribution_params["permit_delay_days"]["Mean"],
        distribution_params["permit_delay_days"]["Std_Dev"]
    )

    equipment = np.random.normal(
        distribution_params["equipment_downtime_hours"]["Mean"],
        distribution_params["equipment_downtime_hours"]["Std_Dev"]
    )

    inflation = np.random.normal(
        distribution_params["inflation_rate_annual"]["Mean"],
        distribution_params["inflation_rate_annual"]["Std_Dev"]
    )

    change = np.random.normal(
        distribution_params["change_orders"]["Mean"],
        distribution_params["change_orders"]["Std_Dev"]
    )

    # same cost model used in Monte Carlo section
    total_cost = (
        baseline_cost
        + material * 5
        - labor * 2
        + weather * 0.3
        + permit * 0.4
        + equipment * 0.05
        + inflation * 4
        + change * 1.5
    )

    simulation_rows.append({
        "material_price_index": material,
        "labor_availability_index": labor,
        "weather_delay_days": weather,
        "permit_delay_days": permit,
        "equipment_downtime_hours": equipment,
        "inflation_rate_annual": inflation,
        "change_orders": change,
        "simulated_total_cost": total_cost
    })

sim_df = pd.DataFrame(simulation_rows)
sim_df.to_csv(output_dir / "all_simulation_inputs_and_costs.csv", index=False)

# ==========================================
# 5. Correlation-based sensitivity analysis
# ==========================================
corr_series = sim_df.corr(numeric_only=True)["simulated_total_cost"].drop("simulated_total_cost")

sensitivity_df = corr_series.reset_index()
sensitivity_df.columns = ["Variable", "Correlation_with_Total_Cost"]
sensitivity_df["Absolute_Importance"] = sensitivity_df["Correlation_with_Total_Cost"].abs()
sensitivity_df = sensitivity_df.sort_values("Absolute_Importance", ascending=False).reset_index(drop=True)
sensitivity_df["Rank"] = range(1, len(sensitivity_df) + 1)

# round values
sensitivity_df["Correlation_with_Total_Cost"] = sensitivity_df["Correlation_with_Total_Cost"].round(4)
sensitivity_df["Absolute_Importance"] = sensitivity_df["Absolute_Importance"].round(4)

# save table
sensitivity_df.to_csv(output_dir / "table_sensitivity_results.csv", index=False)

print("\nSensitivity Analysis Results")
print(sensitivity_df)

# ==========================================
# 6. Plot sensitivity bar chart
# ==========================================
plot_df = sensitivity_df.sort_values("Correlation_with_Total_Cost")

plt.figure(figsize=(9, 6))
plt.barh(plot_df["Variable"], plot_df["Correlation_with_Total_Cost"])
plt.title("Sensitivity Analysis of Monte Carlo Input Variables")
plt.xlabel("Correlation with Simulated Total Cost")
plt.ylabel("Input Variable")
plt.tight_layout()
plt.savefig(output_dir / "figure_sensitivity_bar_chart.png", dpi=300)
plt.close()

print("\nAll outputs saved in:", output_dir)


Sensitivity Analysis Results
                   Variable  Correlation_with_Total_Cost  Absolute_Importance  \
0             change_orders                       0.8071               0.8071   
1         permit_delay_days                       0.3356               0.3356   
2  equipment_downtime_hours                       0.3216               0.3216   
3        weather_delay_days                       0.3198               0.3198   
4      material_price_index                       0.1366               0.1366   
5  labor_availability_index                      -0.0853               0.0853   
6     inflation_rate_annual                       0.0147               0.0147   

   Rank  
0     1  
1     2  
2     3  
3     4  
4     5  
5     6  
6     7  

All outputs saved in: sensitivity_analysis_outputs
